# 🚀 Huấn Luyện AI Gatekeeper v2 (Dữ Liệu Lớn & Chất Lượng Cao) Cho NostalgiaForInfinityX8
### Phiên bản: AI Gatekeeper v2.0 - Đa Khung Thời Gian (MTF) & Nhãn Thực Tế (Realistic Scalp Barrier)

Notebook này được tối ưu hóa đặc biệt để chạy trên **Google Colab GPU (T4 / A100 / V100)** nhằm:
1. **Tải dữ liệu quy mô lớn (Big Data)**: Hỗ trợ kéo trực tiếp 1 - 2 năm lịch sử từ Binance Futures với băng thông 1Gbps của Colab.
2. **Chất lượng đặc trưng cao (Feature Quality 2.0)**: Tích hợp 20 đặc trưng kỹ thuật Đa Khung Thời Gian (5m + 1h + 4h) và độ biến động thực tế (ATR Normalized).
3. **Nhãn thực tế (Realistic Scalp Barrier)**: Chống hiện tượng phạt nhầm lệnh thắng ngắn ($TP \ge 1.0\%$ với tỷ lệ $MFE/MAE \ge 1.3$).
4. **Tối ưu hóa GPU & Optuna**: Tự động tìm kiếm bộ siêu tham số tốt nhất với LightGBM GPU và CatBoost GPU.
5. **Hiệu chuẩn xác suất & Xuất Model chuẩn**: Xuất tệp `nfix8_short_ai_v2.pkl` dạng Dictionary tiêu chuẩn để bot sử dụng ngay.

In [ ]:
# 1. KIỂM TRA PHẦN CỨNG GPU & CÀI ĐẶT THƯ VIỆN
!nvidia-smi

!pip install -q lightgbm catboost optuna pyarrow scikit-learn matplotlib seaborn joblib ccxt shap

In [ ]:
# 2. KẾT NỐI GOOGLE DRIVE (TÙY CHỌN - ĐỂ LƯU MODEL VÀ ĐỌC DATA)
import os
from google.colab import drive
try:
    drive.mount('/content/drive')
    DRIVE_AVAILABLE = True
    DRIVE_BOT_DIR = '/content/drive/MyDrive/botx8' # Điều chỉnh theo thư mục của anh trên Google Drive
    print('✅ Đã kết nối Google Drive thành công!')
except Exception as e:
    DRIVE_AVAILABLE = False
    DRIVE_BOT_DIR = '/content'
    print('⚠️ Chưa mount Drive. Mô hình sẽ được lưu tại Colab local storage.')

### 3. TẢI DỮ LIỆU HUẤN LUYỆN (2 LỰA CHỌN)
- **Lựa chọn A**: Nạp tệp dữ liệu đã trích xuất sạch `ai_dataset_short_v2.parquet` từ máy hoặc Google Drive.
- **Lựa chọn B**: Tải trực tiếp dữ liệu quy mô lớn (1 - 2 năm) từ Binance Futures API ngay trên Colab.

In [ ]:
# 3A. NẠP TỆP DATASET ĐÃ TRÍCH XUẤT SẴN
import pandas as pd
import numpy as np

DATASET_CANDIDATE_PATHS = [
    os.path.join(DRIVE_BOT_DIR, 'user_data/data/ai_dataset_short_v2.parquet'),
    os.path.join(DRIVE_BOT_DIR, 'ai_dataset_short_v2.parquet'),
    'ai_dataset_short_v2.parquet',
    os.path.join(DRIVE_BOT_DIR, 'user_data/data/ai_dataset_short.parquet'),
    'ai_dataset_short.parquet'
]

dataset_path = None
for p in DATASET_CANDIDATE_PATHS:
    if os.path.exists(p):
        dataset_path = p
        break

if dataset_path is None:
    print('⚠️ Chưa tìm thấy file dataset parquet sẵn có. Vui lòng upload file ai_dataset_short_v2.parquet lên Colab hoặc dùng Lựa chọn B!')
    from google.colab import files
    uploaded = files.upload()
    dataset_path = list(uploaded.keys())[0]

print(f'📥 Đang đọc tập dữ liệu từ: {dataset_path}')
df = pd.read_parquet(dataset_path)
if df['date'].dt.tz is None:
    df['date'] = pd.to_datetime(df['date']).dt.tz_localize('UTC')

print(f'🎉 TỔNG SỐ MẪU: {len(df):,} dòng')
print(f'📅 Khoảng thời gian: {df["date"].min()} đến {df["date"].max()}')
print(f'📊 Phân phối nhãn: Win={df["target_win"].sum():,} ({df["target_win"].mean():.1%}) | Noise/Loss={(df["target_win"]==0).sum():,}')
df.head()

In [ ]:
# 4. PHÂN CHIA PURGED TIME-SERIES SPLIT (CHỐNG RÒ RỈ TƯƠNG LAI)
# Danh sách 18 đặc trưng kỹ thuật Đa Khung Thời Gian (MTF)
feature_cols = [
    'upper_wick', 'lower_wick', 'body_ratio',
    'dist_ema21', 'dist_ema50', 'dist_ema200',
    'rsi_14', 'bb_bandwidth', 'atr_norm', 'vol_spike',
    'sell_vol_ratio_3', 'sell_vol_ratio_6',
    'pair_dist_ema50_1h', 'pair_dist_ema200_1h', 'pair_rsi_1h',
    'btc_rsi_1h', 'btc_dist_ema200_1h', 'btc_dist_ema200_4h'
]

# Lọc các cột có sẵn trong dataframe
active_feature_cols = [c for c in feature_cols if c in df.columns]
if not active_feature_cols:
    # Fallback cho dataset v1
    active_feature_cols = [
        'upper_wick', 'lower_wick', 'body_ratio',
        'dist_ema21', 'dist_ema50', 'dist_ema200',
        'rsi_14', 'bb_bandwidth', 'vol_spike', 'sell_vol_ratio',
        'btc_rsi', 'btc_dist_ema200'
    ]

print(f'✅ Sử dụng {len(active_feature_cols)} đặc trưng:')
print(active_feature_cols)

# Sắp xếp tuyệt đối theo dòng thời gian
df = df.sort_values('date').reset_index(drop=True)

# 80% Train, 20% Test (Validation strictly out-of-time)
split_idx = int(len(df) * 0.80)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train, y_train = train_df[active_feature_cols].to_numpy(), train_df['target_win'].to_numpy()
X_test, y_test = test_df[active_feature_cols].to_numpy(), test_df['target_win'].to_numpy()

print(f'Tập Train: {len(X_train):,} mẫu ({train_df["date"].min()} -> {train_df["date"].max()})')
print(f'Tập Test:  {len(X_test):,} mẫu ({test_df["date"].min()} -> {test_df["date"].max()})')

In [ ]:
# 5. TỐI ƯU HÓA SIÊU THAM SỐ VỚI OPTUNA TRÊN GPU (50 TRIALS)
import optuna
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'binary_logloss',
        'boosting_type': 'gbdt',
        'device_type': 'cuda', # Tận dụng GPU Colab
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'max_depth': trial.suggest_int('max_depth', 5, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 30, 200),
        'subsample': trial.suggest_float('subsample', 0.7, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.95),
        'scale_pos_weight': trial.suggest_float('scale_pos_weight', 1.5, 3.5), # Cân bằng chi phí phạt
        'verbose': -1,
        'n_estimators': 250,
        'random_state': 42
    }
    
    clf = lgb.LGBMClassifier(**params)
    clf.fit(X_train, y_train)
    
    preds = clf.predict_proba(X_test)[:, 1]
    # Tối ưu hóa PR-AUC (Average Precision) kết hợp ROC-AUC
    score = 0.5 * roc_auc_score(y_test, preds) + 0.5 * average_precision_score(y_test, preds)
    return score

print('🚀 Bắt đầu tối ưu 50 trials với Optuna trên GPU...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=600)

print('🏆 Best Combined Score:', study.best_value)
print('🏆 Best Params:', study.best_params)

In [ ]:
# 6. HUẤN LUYỆN MÔ HÌNH CHÍNH THỨC & HIỆU CHUẨN XÁC SUẤT PERCENTILE
from sklearn.metrics import precision_score

best_params = study.best_params
best_params.update({
    'objective': 'binary',
    'metric': 'binary_logloss',
    'device_type': 'cuda',
    'n_estimators': 350,
    'verbose': -1,
    'random_state': 42
})

model = lgb.LGBMClassifier(**best_params)
model.fit(X_train, y_train)

# Trích xuất phân vị tập Train để tạo Engine hiệu chuẩn Percentile
raw_train_p = model.predict_proba(X_train)[:, 1]
p25 = float(np.percentile(raw_train_p, 25))
p50 = float(np.percentile(raw_train_p, 50))
p75 = float(np.percentile(raw_train_p, 75))

print(f'📊 Phân vị xác suất tập Train: P25={p25:.4f}, P50={p50:.4f}, P75={p75:.4f}')

# Hàm chấm điểm hiệu chuẩn Percentile sang thang điểm [0.10, 0.95]
def calibrate_scores(raw_p, p25_val, p50_val, p75_val):
    scores = np.interp(raw_p, [0.05, p25_val, p50_val, p75_val, 0.40], [0.10, 0.40, 0.60, 0.75, 0.95])
    return np.clip(scores, 0.05, 0.98)

raw_test_p = model.predict_proba(X_test)[:, 1]
test_scores = calibrate_scores(raw_test_p, p25, p50, p75)

print(f'🎯 Test ROC-AUC: {roc_auc_score(y_test, test_scores):.4f}')
print(f'🎯 Test PR-AUC:  {average_precision_score(y_test, test_scores):.4f}')

print('\n📈 HIỆU SUẤT TRÊN TẬP TEST SAU KHI HIỆU CHUẨN THANG ĐIỂM [0.0 - 1.0]:')
print(f'{"Ngưỡng AI (Threshold)":<22} | {"Số lệnh duyệt":<14} | {"Tỷ lệ duyệt":<12} | {"Tỷ lệ thắng (Precision)":<25}')
print('-' * 80)
for th in [0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
    app = test_scores >= th
    if app.sum() > 0:
        prec = precision_score(y_test[app], np.ones(app.sum()))
        print(f'Điểm tin cậy >= {th:.2f}     | {app.sum():>10,}     | {app.mean():>10.1%} | {prec:>20.1%}')

In [ ]:
# 7. PHÂN TÍCH ĐẶC TRƯNG QUAN TRỌNG NHẤT (FEATURE IMPORTANCE & SHAP)
import matplotlib.pyplot as plt
import seaborn as sns

importances = model.feature_importances_
feat_df = pd.DataFrame({'feature': active_feature_cols, 'importance': importances}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 7))
sns.barplot(data=feat_df, x='importance', y='feature', palette='viridis')
plt.title('Top Đặc Trưng Quyết Định Của AI Gatekeeper v2')
plt.tight_layout()
plt.show()

In [ ]:
# 8. XUẤT MÔ HÌNH CHUẨN DICTIONARY VÀO BOT
import joblib, time

# Chuyển model về CPU để máy local bot không cần cài CUDA vẫn dự đoán siêu tốc (0.1ms)
model.set_params(device_type='cpu')

model_payload = {
    'model': model,
    'p25': p25,
    'p50': p50,
    'p75': p75,
    'feature_cols': active_feature_cols,
    'version': 'v2.0_MTF_RealisticBarrier',
    'trained_at': time.strftime('%Y-%m-%d %H:%M:%S')
}

LOCAL_OUT_PATH = 'nfix8_short_ai_v2.pkl'
joblib.dump(model_payload, LOCAL_OUT_PATH)
print(f'💾 Đã lưu tệp mô hình v2 tại Colab: {LOCAL_OUT_PATH} ({os.path.getsize(LOCAL_OUT_PATH)/1024:.1f} KB)')

if DRIVE_AVAILABLE:
    DRIVE_OUT_PATH = os.path.join(DRIVE_BOT_DIR, 'user_data/models/nfix8_short_ai_v2.pkl')
    os.makedirs(os.path.dirname(DRIVE_OUT_PATH), exist_ok=True)
    joblib.dump(model_payload, DRIVE_OUT_PATH)
    print(f'💾 Đã đồng bộ trực tiếp vào Google Drive của bot: {DRIVE_OUT_PATH}')

print('\n🎉 HOÀN TẤT HUẤN LUYỆN! Bây giờ anh có thể nạp nfix8_short_ai_v2.pkl vào bot để backtest!')

# Tải trực tiếp về máy tính (Nếu không dùng Google Drive)
from google.colab import files
files.download(LOCAL_OUT_PATH)